<a href="https://colab.research.google.com/github/cruhling289/MSDSCapstone/blob/main/notebooks/MSDSCapstone.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

**Creating text variable with the content**

In [5]:
#config.py
k = 2
chunk_size = 1000
overlap = 200
#max_new_tokens=300

In [6]:
!wget -q https://raw.githubusercontent.com/cruhling289/MSDSCapstone/main/data/capstone_day_planning.md
with open("capstone_day_planning.md", "r") as f:
    text1 = f.read()

#print(text[:2000])

**another way to read the text**

In [3]:
import requests
try:
  rawURL = "https://raw.githubusercontent.com/cruhling289/MSDSCapstone/main/data/capstone_day_planning.md"
  response = requests.get(rawURL, timeout=10)
  response.raise_for_status
  text = response.text
except Exception as e:
  print(f'Unexpected error: {e}')

#print(text)

In [4]:
import requests

url = "https://datascience.virginia.edu/data-science-capstone/sponsor-experience"

response = requests.get(url)

print(response.status_code)

200


In [7]:
import requests

try:
  rawURL = "https://datascience.virginia.edu/data-science-capstone/sponsor-experience"
  response = requests.get(rawURL, timeout=10)
  response.raise_for_status()
  web_text = response.text
except Exception as e:
  print(f'Unexpected error: {e}')


from bs4 import BeautifulSoup

soup = BeautifulSoup(web_text, "html.parser")
web_text = soup.get_text(separator="\n")

#print(web_text[:2000])

**Combining the two texts**

In [8]:
text = text1 + "\n\n" + web_text

print(len(text))

13215


**Auto-chunking**

In [9]:
chunkList = []

for i in range(0, len(text), chunk_size - overlap):
    chunk = text[i:i + chunk_size]
    chunkList.append(chunk)

#print(len(chunkList))

**Manually chunking text, fix later to make it automatic**

In [ ]:
chunks = [
    "## Event and Venue Info",
    "## Preparing for the Event",
    "## Attendance",
    "## Awards",
    "## Presentation Schedule",
]
chunkList = []
currentChunk = ""

for line in text.splitlines():

    if line in chunks:
        if currentChunk != "":
          chunkList.append(currentChunk)
        currentChunk = line + "\n"
    else:
        currentChunk += line + "\n"
chunkList.append(currentChunk)




In [ ]:
!pip install -q sentence-transformers
from sentence_transformers import SentenceTransformer

**Embedding chunked text**

In [ ]:
embedding_model = SentenceTransformer("all-MiniLM-L6-v2")
#embedding_practice = embedding_model.encode("Which talks happen before 11AM")
#print(embedding_practice)
#print(len(embedding_practice))
embeddings = []
for chunk in chunkList:
  x = embedding_model.encode(chunk)
  embeddings.append(x)
#print(len(embeddings))
#print(len(embeddings[0]))

In [ ]:
from sklearn.metrics.pairwise import cosine_similarity

**Vector similarity/retrieval**

In [ ]:
query = input("What is your question? ")
queryEmbedding = embedding_model.encode(query)
similarities = []
for embedding in embeddings:
  similarity = cosine_similarity([queryEmbedding], [embedding]) [0][0]
  similarities.append(similarity)
#print((similarities))

top_indices = sorted(
    range(len(similarities)),
    key=lambda i: similarities[i],
    reverse=True
)[:k]

retrievedChunks = []
for i in top_indices:
  retrievedChunks.append(chunkList[i])

**Combining the k retrieved chunks**

In [ ]:
joinedRetrieval = ""
for i in range(len(retrievedChunks)):
  joinedRetrieval += retrievedChunks[i] + "\n\n"


**Generation**

In [ ]:
prompt = f'''Using only the following context, answer the question.
  Context: {joinedRetrieval} \n
  Question: {query}\n
  Answer: '''
#print(prompt)

**importing the llm (qwen3) through the transformers library**

In [ ]:
from transformers import AutoTokenizer, AutoModelForCausalLM

tokenizer = AutoTokenizer.from_pretrained("Qwen/Qwen3-1.7B")
model = AutoModelForCausalLM.from_pretrained("Qwen/Qwen3-1.7B", device_map="auto")
messages = [
    {"role": "user", "content": prompt},
]
inputs = tokenizer.apply_chat_template(
	messages,
	add_generation_prompt=True,
	tokenize=True,
	return_dict=True,
	return_tensors="pt",
	enable_thinking=False,
).to(model.device)

outputs = model.generate(**inputs, max_new_tokens=300)
print(tokenizer.decode(outputs[0][inputs["input_ids"].shape[-1]:]))

config.json:   0%|          | 0.00/726 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/9.73k [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/2.78M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/1.67M [00:00<?, ?B/s]

tokenizer.json: reconstructing file:   0%|          |  0.00B / 11.4MB            

...

Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/311 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/239 [00:00<?, ?B/s]

You need to arrive by **9:25am** at the lawn side of the Rotunda.<|im_end|>
